In [1]:
# ============================================================================
# notebook: notebooks/07_robustness.ipynb
# Project: "Incidental vs. Engineered Approval"
# Stage 5a: bootstrap confidence intervals for the headline numbers, so the
#   paper's Results table rests on interval estimates, not point estimates.
#   Targets:
#     R1. V4 convergent validity: EngineeredScore->default coef (borderline)
#     R2. RQ2 composite gap (dis - adv) on EngineeredScore  (expect ~0)
#     R3. RQ2 sub-axis gaps (Stability, NonFragility, LowDensity)  (robust)
#     R4. typicality paradox: density->default coef (borderline)
#   Method: nonparametric bootstrap (resample rows with replacement, B draws).
# Reads results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, load scored borderline set
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
B_BOOT = 2000

B = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
B["density_pct"] = 1.0 - B["A_LowDensity"]
dis = B[B["GROUP"]=="dis_primary"].copy()
adv = B[B["GROUP"]=="advantaged"].copy()
print(f"Borderline: {len(B)}  | dis_primary={len(dis)} adv={len(adv)}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Helpers: bootstrap a statistic with percentile CI
# ─────────────────────────────────────────────────────────────────────────
def boot_ci(fn, *frames, B=B_BOOT, seed=RANDOM_STATE):
    """Resample each frame with replacement, apply fn, return (point, lo, hi, vals)."""
    r = np.random.default_rng(seed)
    point = fn(*frames)
    vals = np.empty(B)
    for b in range(B):
        resampled = [f.sample(len(f), replace=True, random_state=int(r.integers(1e9))) for f in frames]
        vals[b] = fn(*resampled)
    lo, hi = np.nanpercentile(vals, [2.5, 97.5])
    return point, lo, hi, vals

def logit_coef(frame, xcol):
    X = sm.add_constant(frame[[xcol, "P_VIP"]])
    m = sm.Logit(frame["DEFAULT"].values, X).fit(disp=0)
    return m.params[xcol]

def gap(a, b, col):
    return a[col].mean() - b[col].mean()


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — R1: V4 convergent validity coef (EngineeredScore -> default)
# ─────────────────────────────────────────────────────────────────────────
p, lo, hi, _ = boot_ci(lambda f: logit_coef(f, "EngineeredScore"), B)
print("R1 — V4 convergent validity (ES->default coef | p(x), borderline):")
print(f"  coef = {p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
      f"{'excludes 0' if hi<0 or lo>0 else 'includes 0'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — R2 & R3: composite + sub-axis group gaps (dis - adv)
# ─────────────────────────────────────────────────────────────────────────
print("\nR2/R3 — group gaps (dis_primary - advantaged), bootstrap CI:")
gaps = {}
for col in ["EngineeredScore", "A_Stability", "A_NonFrag", "A_LowDensity"]:
    p, lo, hi, _ = boot_ci(lambda a, b: gap(a, b, col), dis, adv)
    excl = "excludes 0" if (hi < 0 or lo > 0) else "includes 0"
    gaps[col] = (p, lo, hi, excl)
    print(f"  {col:16} gap={p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  {excl}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — R4: typicality paradox coef (density -> default)
# ─────────────────────────────────────────────────────────────────────────
p, lo, hi, _ = boot_ci(lambda f: logit_coef(f, "density_pct"), B)
print("\nR4 — typicality paradox (density->default coef | p(x), borderline):")
print(f"  coef = {p:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
      f"{'excludes 0 (paradox robust)' if lo>0 else 'includes 0'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Assemble the paper's headline Results table + save
# ─────────────────────────────────────────────────────────────────────────
rows = [
    ("V4 convergent (ES->default)", *boot_ci(lambda f: logit_coef(f,"EngineeredScore"), B)[:3]),
    ("Paradox (density->default)",  *boot_ci(lambda f: logit_coef(f,"density_pct"), B)[:3]),
    ("Gap composite (ES)",          *boot_ci(lambda a,b: gap(a,b,"EngineeredScore"), dis, adv)[:3]),
    ("Gap Stability",               *boot_ci(lambda a,b: gap(a,b,"A_Stability"), dis, adv)[:3]),
    ("Gap NonFragility",            *boot_ci(lambda a,b: gap(a,b,"A_NonFrag"), dis, adv)[:3]),
    ("Gap LowDensity",              *boot_ci(lambda a,b: gap(a,b,"A_LowDensity"), dis, adv)[:3]),
]
tab = pd.DataFrame(rows, columns=["quantity","point","ci_lo","ci_hi"]).round(3)
tab["excludes_0"] = (tab["ci_hi"]<0) | (tab["ci_lo"]>0)
print("\n=== HEADLINE RESULTS TABLE (bootstrap 95% CI) ===")
print(tab.to_string(index=False))
tab.to_csv(RESULTS / "stage5_bootstrap_ci.csv", index=False)
print("\nSaved -> results/stage5_bootstrap_ci.csv")


# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Robustness verdict
# ─────────────────────────────────────────────────────────────────────────
print("=" * 66)
print("STAGE 5a — BOOTSTRAP ROBUSTNESS VERDICT")
print("=" * 66)
v4  = tab[tab.quantity.str.startswith("V4")].iloc[0]
par = tab[tab.quantity.str.startswith("Paradox")].iloc[0]
gES = tab[tab.quantity=="Gap composite (ES)"].iloc[0]
gS  = tab[tab.quantity=="Gap Stability"].iloc[0]
gF  = tab[tab.quantity=="Gap NonFragility"].iloc[0]
print(f"V4 convergent      : {'robust' if v4.excludes_0 else 'uncertain'}")
print(f"Paradox            : {'robust' if par.excludes_0 else 'uncertain'}")
print(f"Composite gap ~0   : {'CI includes 0 (as expected)' if not gES.excludes_0 else 'nonzero'}")
print(f"Stability gap      : {'robust' if gS.excludes_0 else 'uncertain'}")
print(f"NonFragility gap   : {'robust' if gF.excludes_0 else 'uncertain'}")
print("-" * 66)
print("Interpretation: the composite gap CI should straddle 0 (equal overall),")
print("while Stability & NonFragility gaps should exclude 0 (different composition).")
print("=" * 66)

Borderline: 1141  | dis_primary=254 adv=224
R1 — V4 convergent validity (ES->default coef | p(x), borderline):
  coef = -1.249  95% CI [-2.520, -0.109]  excludes 0

R2/R3 — group gaps (dis_primary - advantaged), bootstrap CI:
  EngineeredScore  gap=-0.006  95% CI [-0.034, +0.021]  includes 0
  A_Stability      gap=+0.086  95% CI [+0.045, +0.125]  excludes 0
  A_NonFrag        gap=-0.072  95% CI [-0.110, -0.033]  excludes 0
  A_LowDensity     gap=-0.033  95% CI [-0.083, +0.019]  includes 0

R4 — typicality paradox (density->default coef | p(x), borderline):
  coef = +0.976  95% CI [+0.332, +1.639]  excludes 0 (paradox robust)

=== HEADLINE RESULTS TABLE (bootstrap 95% CI) ===
                   quantity  point  ci_lo  ci_hi  excludes_0
V4 convergent (ES->default) -1.249 -2.520 -0.109        True
 Paradox (density->default)  0.976  0.332  1.639        True
         Gap composite (ES) -0.006 -0.034  0.021       False
              Gap Stability  0.086  0.045  0.125        True
           